# Debug QIN Prostate Repeatability NIfTI Conversion

Investigate potential spatial mismatch between converted T2 MR volumes and wholegland segmentations in `index.csv`.

## 1. Setup

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DATA_ROOT = Path("../data/raw/tcia/qin_prostate_repeatability")
index = pd.read_csv(DATA_ROOT / "index.csv")
display(index.head())
print(f"{len(index)} cases")

,ID,t2,t2_wholegland
0,PCAMPMRI-00001_19940620,mr_nii/PCAMPMRI-00001_19940620_t2_weighted_axi...,seg_nii/PCAMPMRI-00001_19940620_t2_weighted_ax...
1,PCAMPMRI-00001_19940702,mr_nii/PCAMPMRI-00001_19940702_t2_weighted_axi...,seg_nii/PCAMPMRI-00001_19940702_t2_weighted_ax...
2,PCAMPMRI-00002_19940709,mr_nii/PCAMPMRI-00002_19940709_t2_weighted_axi...,seg_nii/PCAMPMRI-00002_19940709_t2_weighted_ax...
3,PCAMPMRI-00002_19940723,mr_nii/PCAMPMRI-00002_19940723_t2_weighted_axi...,seg_nii/PCAMPMRI-00002_19940723_t2_weighted_ax...
4,PCAMPMRI-00003_19940811,mr_nii/PCAMPMRI-00003_19940811_t2_weighted_axi...,seg_nii/PCAMPMRI-00003_19940811_t2_weighted_ax...


30 cases


## 2. Header Comparison — All Cases

Load both NIfTI headers and compare shape, spacing, and affine origin.

In [24]:
rows = []
for _, row in index.iterrows():
    img_path = DATA_ROOT / row["t2"]
    seg_path = DATA_ROOT / row["t2_wholegland"]

    img_exists = img_path.exists()
    seg_exists = seg_path.exists()

    if not img_exists or not seg_exists:
        rows.append({"id": row["ID"], "img_exists": img_exists, "seg_exists": seg_exists,
                     "shape_match": False, "origin_diff_mm": None, "spacing_match": None})
        continue

    img_nii = nib.load(img_path)
    seg_nii = nib.load(seg_path)

    img_shape = img_nii.shape
    seg_shape = seg_nii.shape
    img_zooms = np.array(img_nii.header.get_zooms()[:3])
    seg_zooms = np.array(seg_nii.header.get_zooms()[:3])
    img_origin = img_nii.affine[:3, 3]
    seg_origin = seg_nii.affine[:3, 3]

    rows.append({
        "id": row["ID"],
        "img_exists": img_exists,
        "seg_exists": seg_exists,
        "img_shape": img_shape,
        "seg_shape": seg_shape,
        "shape_match": img_shape == seg_shape,
        "img_spacing": np.round(img_zooms, 3).tolist(),
        "seg_spacing": np.round(seg_zooms, 3).tolist(),
        "spacing_match": np.allclose(img_zooms, seg_zooms, atol=0.01),
        "img_origin": np.round(img_origin, 2).tolist(),
        "seg_origin": np.round(seg_origin, 2).tolist(),
        "origin_diff_mm": np.round(np.linalg.norm(img_origin - seg_origin), 2),
    })

hdr_df = pd.DataFrame(rows)
display(hdr_df[["id", "img_exists", "seg_exists", "img_shape", "seg_shape",
                 "shape_match", "img_spacing", "seg_spacing", "spacing_match", "origin_diff_mm"]])

n_shape_mismatch = (~hdr_df["shape_match"]).sum()
n_large_origin_diff = (hdr_df["origin_diff_mm"].fillna(0) > 5).sum()
print(f"Shape mismatches: {n_shape_mismatch}/{len(hdr_df)}")
print(f"Origin diff > 5 mm: {n_large_origin_diff}/{len(hdr_df)}")

,id,img_exists,seg_exists,img_shape,seg_shape,shape_match,img_spacing,seg_spacing,spacing_match,origin_diff_mm
0,PCAMPMRI-00001_19940620,True,True,"(512, 512, 30)","(512, 512, 30)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
1,PCAMPMRI-00001_19940702,True,True,"(512, 512, 30)","(512, 512, 30)",True,"[0.27300000190734863, 0.27300000190734863, 3.0]","[0.27300000190734863, 0.27300000190734863, 3.0]",True,0.0
2,PCAMPMRI-00002_19940709,True,True,"(512, 512, 30)","(512, 512, 30)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
3,PCAMPMRI-00002_19940723,True,True,"(512, 512, 30)","(512, 512, 30)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
4,PCAMPMRI-00003_19940811,True,True,"(512, 512, 30)","(512, 512, 30)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
5,PCAMPMRI-00003_19940825,True,True,"(512, 512, 32)","(512, 512, 32)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
6,PCAMPMRI-00004_19940903,True,True,"(512, 512, 30)","(512, 512, 30)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
7,PCAMPMRI-00004_19940912,True,True,"(512, 512, 32)","(512, 512, 32)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
8,PCAMPMRI-00005_19940910,True,True,"(512, 512, 32)","(512, 512, 32)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0
9,PCAMPMRI-00005_19940917,True,True,"(512, 512, 34)","(512, 512, 34)",True,"[0.31200000643730164, 0.31200000643730164, 3.0]","[0.31200000643730164, 0.31200000643730164, 3.0]",True,0.0


Shape mismatches: 0/30
Origin diff > 5 mm: 0/30


## 3. Visual Alignment Check

Axial overlays: T2 greyscale + wholegland mask (red, 40% alpha). Three slices per case (top-third / mid / bottom-third of mask extent).

In [ ]:
def pick_slices(mask_3d):
    """Return three axial slice indices spanning the mask extent (or evenly spaced if no mask)."""
    z_has_mask = np.where(mask_3d.any(axis=(0, 1)))[0]
    if len(z_has_mask) == 0:
        n = mask_3d.shape[2]
        return [n // 4, n // 2, 3 * n // 4]
    lo, hi = z_has_mask[0], z_has_mask[-1]
    mid = (lo + hi) // 2
    return [lo + (mid - lo) // 2, mid, mid + (hi - mid) // 2]


def overlay_slice(ax, img_slice, seg_slice, title=""):
    p2, p98 = np.percentile(img_slice, [2, 98])
    img_norm = np.clip((img_slice - p2) / (p98 - p2 + 1e-8), 0, 1)
    ax.imshow(img_norm.T, cmap="gray", origin="lower", aspect="equal")
    if seg_slice.any():
        rgba = np.zeros((*seg_slice.shape, 4))
        rgba[..., 0] = 1.0   # red
        rgba[..., 3] = seg_slice * 0.4
        ax.imshow(rgba.transpose(1, 0, 2), origin="lower", aspect="equal")
    ax.set_title(title, fontsize=7)
    ax.axis("off")


valid = hdr_df[hdr_df["img_exists"] & hdr_df["seg_exists"]]
n_cases = len(valid)
fig, axes = plt.subplots(n_cases, 3, figsize=(9, 3 * n_cases))
if n_cases == 1:
    axes = axes[np.newaxis]

for ax_row, (_, meta) in zip(axes, valid.iterrows()):
    case_id = meta["id"]
    img_arr = nib.load(DATA_ROOT / index.loc[index["ID"] == case_id, "t2"].values[0]).get_fdata()
    seg_arr = nib.load(DATA_ROOT / index.loc[index["ID"] == case_id, "t2_wholegland"].values[0]).get_fdata()
    seg_bin = (seg_arr > 0.5).astype(np.uint8)
    slices = pick_slices(seg_bin if seg_bin.shape == img_arr.shape else np.zeros_like(img_arr))
    for ax, z in zip(ax_row, slices):
        img_sl = img_arr[:, :, z] if img_arr.ndim == 3 else img_arr[:, :, z, 0]
        seg_sl = seg_bin[:, :, z] if seg_bin.shape == img_arr.shape else np.zeros(img_arr.shape[:2])
        overlay_slice(ax, img_sl, seg_sl, title=f"{case_id}\nz={z}")

plt.tight_layout()
plt.show()

## 4. Affine Deep-Dive — Representative Case

Compare image vs. segmentation affines for the first case.

In [26]:
first = index.iloc[0]
img_nii = nib.load(DATA_ROOT / first["t2"])
seg_nii = nib.load(DATA_ROOT / first["t2_wholegland"])

np.set_printoptions(precision=4, suppress=True)
print(f"Case: {first['ID']}")
print(f"\nImage shape:  {img_nii.shape}   spacing: {np.round(img_nii.header.get_zooms()[:3], 3)}")
print(f"Seg shape:    {seg_nii.shape}   spacing: {np.round(seg_nii.header.get_zooms()[:3], 3)}")
print(f"\n--- Image affine ---\n{img_nii.affine}")
print(f"\n--- Seg affine ---\n{seg_nii.affine}")
print(f"\n--- Difference (img - seg) ---\n{img_nii.affine - seg_nii.affine}")
print(f"\nMax absolute difference: {np.abs(img_nii.affine - seg_nii.affine).max():.4f}")

Case: PCAMPMRI-00001_19940620

Image shape:  (512, 512, 30)   spacing: [0.312 0.312 3.   ]
Seg shape:    (512, 512, 30)   spacing: [0.312 0.312 3.   ]

--- Image affine ---
[[ -0.312    0.0169  -0.0219  77.7007]
 [ -0.017   -0.3117   0.1485  89.4624]
 [ -0.0014   0.0156   2.9962 -50.962 ]
 [  0.       0.       0.       1.    ]]

--- Seg affine ---
[[ -0.312    0.0169  -0.0219  77.7007]
 [ -0.017   -0.3117   0.1485  89.4624]
 [ -0.0014   0.0156   2.9962 -50.962 ]
 [  0.       0.       0.       1.    ]]

--- Difference (img - seg) ---
[[ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]
 [-0.  0.  0.  0.]
 [ 0.  0.  0.  0.]]

Max absolute difference: 0.0000


In [27]:
# Check all cases for affine agreement
aff_rows = []
for _, row in index.iterrows():
    img_path = DATA_ROOT / row["t2"]
    seg_path = DATA_ROOT / row["t2_wholegland"]
    if not img_path.exists() or not seg_path.exists():
        continue
    diff = nib.load(img_path).affine - nib.load(seg_path).affine
    aff_rows.append({"id": row["ID"], "max_affine_diff": np.abs(diff).max().round(4)})

aff_df = pd.DataFrame(aff_rows)
display(aff_df)
print(f"Affine perfectly matched: {(aff_df['max_affine_diff'] == 0).sum()}/{len(aff_df)}")
print(f"Max affine diff across all cases: {aff_df['max_affine_diff'].max()}")

,id,max_affine_diff
0,PCAMPMRI-00001_19940620,0.0
1,PCAMPMRI-00001_19940702,0.0
2,PCAMPMRI-00002_19940709,0.0
3,PCAMPMRI-00002_19940723,0.0
4,PCAMPMRI-00003_19940811,0.0
5,PCAMPMRI-00003_19940825,0.0
6,PCAMPMRI-00004_19940903,0.0
7,PCAMPMRI-00004_19940912,0.0
8,PCAMPMRI-00005_19940910,0.0
9,PCAMPMRI-00005_19940917,0.0


Affine perfectly matched: 30/30
Max affine diff across all cases: 0.0


## 5. Raw SEG NIfTI Inventory

Cross-reference all `seg_nii/` files against what `index.csv` expects.

In [28]:
seg_nii_dir = DATA_ROOT / "seg_nii"
all_seg_files = sorted(seg_nii_dir.glob("*.nii.gz"))
print(f"Total seg_nii files: {len(all_seg_files)}")

# Files expected by index.csv
expected = set(index["t2_wholegland"].apply(lambda p: (DATA_ROOT / p).name))
found = {f.name for f in all_seg_files}

missing = expected - found
extra = found - expected

print(f"\nExpected by index.csv: {len(expected)}")
print(f"Found in seg_nii/:     {len(found)}")
print(f"Missing:               {len(missing)}")
print(f"Extra (not in index):  {len(extra)}")

if missing:
    print("\n-- Missing files --")
    for f in sorted(missing):
        print(" ", f)

if extra:
    print("\n-- Extra seg files (by segment type) --")
    seg_types = pd.Series([f.replace(".nii.gz", "").split("_")[-1] for f in found])
    display(seg_types.value_counts().rename("count").to_frame())

Total seg_nii files: 326

Expected by index.csv: 30
Found in seg_nii/:     326
Missing:               0
Extra (not in index):  296

-- Extra seg files (by segment type) --


,count
1,146
peripheralzone,90
wholegland,90


## 6. Raw DICOM vs Converted NIfTI — Direct Pixel Comparison

Since shapes and affines agree, the mismatch must be in pixel data ordering. Load the raw DICOM SEG and MR for case 1 and visualize them directly, then compare with the NIfTI versions to find which axis is flipped.

In [29]:
import pydicom, SimpleITK as sitk

IMAGES_DIR = DATA_ROOT / "images"

# --- Find the T2 MR and SEG series dirs for case 1 ---
case1 = index.iloc[0]  # PCAMPMRI-00001_19940620

mr_series_dir = seg_series_dir = None
for d in sorted(IMAGES_DIR.iterdir()):
    dcms = sorted(d.glob("*.dcm"))
    if not dcms:
        continue
    ds = pydicom.dcmread(str(dcms[0]), stop_before_pixels=True)
    pid  = str(ds.get("PatientID", "")).strip()
    date = str(ds.get("StudyDate", "")).strip()
    mod  = str(ds.get("Modality", "")).strip()
    desc = str(ds.get("SeriesDescription", "")).strip()
    if pid != "PCAMPMRI-00001" or date != "19940620":
        continue
    if mod == "MR" and "T2 Weighted Axial" in desc:
        mr_series_dir = d
    if mod == "SEG":
        seg_series_dir = d

print("MR  dir:", mr_series_dir)
print("SEG dir:", seg_series_dir)

MR  dir: ../data/raw/tcia/qin_prostate_repeatability/images/1.3.6.1.4.1.14519.5.2.1.3671.4754.983460207615355998147518323000
SEG dir: ../data/raw/tcia/qin_prostate_repeatability/images/1.2.276.0.7230010.3.1.3.1426846371.6596.1513205116.412


In [30]:
# --- Load MR slices directly from DICOM (sorted by InstanceNumber) ---
mr_dcms = sorted(mr_series_dir.glob("*.dcm"),
                 key=lambda f: int(pydicom.dcmread(str(f), stop_before_pixels=True).InstanceNumber))
mr_raw = np.stack([pydicom.dcmread(str(f)).pixel_array for f in mr_dcms], axis=0)  # (D, H, W)
mr_ipps = [np.array([float(v) for v in pydicom.dcmread(str(f), stop_before_pixels=True).ImagePositionPatient])
           for f in mr_dcms]
print(f"MR DICOM shape: {mr_raw.shape}  (D, H, W)")
print(f"Slice positions (z component): {[round(p[2],1) for p in mr_ipps]}")

MR DICOM shape: (30, 512, 512)  (D, H, W)
Slice positions (z component): [np.float64(35.9), np.float64(32.9), np.float64(29.9), np.float64(26.9), np.float64(23.9), np.float64(20.9), np.float64(18.0), np.float64(15.0), np.float64(12.0), np.float64(9.0), np.float64(6.0), np.float64(3.0), np.float64(-0.0), np.float64(-3.0), np.float64(-6.0), np.float64(-9.0), np.float64(-12.0), np.float64(-15.0), np.float64(-18.0), np.float64(-21.0), np.float64(-24.0), np.float64(-27.0), np.float64(-30.0), np.float64(-33.0), np.float64(-36.0), np.float64(-39.0), np.float64(-42.0), np.float64(-45.0), np.float64(-48.0), np.float64(-51.0)]


In [31]:
# --- Load SEG DICOM: extract WholeGland segment per frame, record IPP ---
seg_file = next(seg_series_dir.glob("*.dcm"))
seg_ds = pydicom.dcmread(str(seg_file))
pffg = seg_ds.PerFrameFunctionalGroupsSequence
num_to_desc = {int(s.SegmentNumber): s.SegmentDescription for s in seg_ds.SegmentSequence}
print("Segments:", dict(num_to_desc))

wg_num = next(n for n, d in num_to_desc.items() if "Whole" in str(d) or "whole" in str(d).lower())
print(f"WholeGland segment number: {wg_num}")

seg_pixels = seg_ds.pixel_array  # (n_frames, H, W)
seg_frames = []
for i, frame in enumerate(pffg):
    sn = int(frame.SegmentIdentificationSequence[0].ReferencedSegmentNumber)
    ipp = np.array([float(v) for v in frame.PlanePositionSequence[0].ImagePositionPatient])
    seg_frames.append((sn, ipp, seg_pixels[i]))

# Sort WholeGland frames by z position (same order as MR)
wg_frames = [(ipp, pix) for sn, ipp, pix in seg_frames if sn == wg_num]
wg_frames.sort(key=lambda x: x[0][2])
wg_ipps  = [x[0] for x in wg_frames]
wg_stack = np.stack([x[1] for x in wg_frames], axis=0)  # (D, H, W)
print(f"SEG WholeGland stack shape: {wg_stack.shape}")
print(f"SEG slice positions (z): {[round(p[2],1) for p in wg_ipps]}")

Segments: {1: 'NormalROI_PZ_1', 2: 'PeripheralZone', 3: 'TumorROI_PZ_1', 4: 'WholeGland'}
WholeGland segment number: 4
SEG WholeGland stack shape: (16, 256, 256)
SEG slice positions (z): [np.float64(-54.7), np.float64(-48.7), np.float64(-42.7), np.float64(-36.7), np.float64(-30.7), np.float64(-24.7), np.float64(-18.8), np.float64(-12.8), np.float64(-6.8), np.float64(-0.8), np.float64(5.2), np.float64(11.2), np.float64(17.2), np.float64(23.2), np.float64(29.2), np.float64(35.2)]


In [32]:
# --- Compare MR z-positions vs SEG z-positions ---
# MR and SEG should have matching IPPs for corresponding slices
mr_zs  = [round(p[2], 2) for p in mr_ipps]
seg_zs = [round(p[2], 2) for p in wg_ipps]
match_df = pd.DataFrame({"mr_z": mr_zs, "seg_z": seg_zs[:len(mr_zs)] if len(seg_zs) >= len(mr_zs) else seg_zs + [None]*(len(mr_zs)-len(seg_zs))})
match_df["z_diff"] = (match_df["mr_z"] - match_df["seg_z"]).abs()
display(match_df)
print(f"Z positions matched: {(match_df['z_diff'] < 0.1).all()}")

,mr_z,seg_z,z_diff
0,35.93,-54.71,90.64
1,32.93,-48.71,81.64
2,29.94,-42.72,72.66
3,26.94,-36.73,63.67
4,23.94,-30.74,54.68
5,20.95,-24.74,45.69
6,17.95,-18.75,36.70
7,14.96,-12.76,27.72
8,11.96,-6.77,18.73
9,8.96,-0.77,9.73


Z positions matched: False


In [ ]:
# --- Side-by-side: raw DICOM overlay vs converted NIfTI overlay ---
# Pick 4 slices where the SEG has content
wg_active_slices = np.where(wg_stack.any(axis=(1, 2)))[0]
show_slices = wg_active_slices[np.linspace(0, len(wg_active_slices)-1, 4, dtype=int)]

img_nii_arr = nib.load(DATA_ROOT / case1["t2"]).get_fdata()           # (512,512,D) nibabel RAS
seg_nii_arr = nib.load(DATA_ROOT / case1["t2_wholegland"]).get_fdata()

fig, axes = plt.subplots(3, len(show_slices), figsize=(3*len(show_slices), 9))
for col, z in enumerate(show_slices):
    # Row 0: raw DICOM MR + raw SEG (DICOM array: D,H,W → show H,W at slice z)
    mr_sl  = mr_raw[z]          # (H, W)
    seg_sl = wg_stack[z]        # (H, W)
    p2, p98 = np.percentile(mr_sl, [2, 98])
    axes[0, col].imshow(np.clip((mr_sl-p2)/(p98-p2+1e-8),0,1), cmap="gray", origin="upper")
    if seg_sl.any():
        rgba = np.zeros((*seg_sl.shape, 4)); rgba[...,0]=1; rgba[...,3]=seg_sl*0.4
        axes[0, col].imshow(rgba, origin="upper")
    axes[0, col].set_title(f"DICOM raw  z={z}", fontsize=8)
    axes[0, col].axis("off")

    # Row 1: NIfTI MR + NIfTI SEG (nibabel: shape H,W,D with axes transposed for display)
    # nibabel RAS convention: arr[i,j,k] where k=slice; display as arr[:,:,z].T for imshow
    nii_mr_sl  = img_nii_arr[:, :, z]   # (512,512)
    nii_seg_sl = (seg_nii_arr[:, :, z] > 0.5).astype(np.uint8)
    p2, p98 = np.percentile(nii_mr_sl, [2, 98])
    axes[1, col].imshow(np.clip((nii_mr_sl-p2)/(p98-p2+1e-8),0,1).T, cmap="gray", origin="lower")
    if nii_seg_sl.any():
        rgba = np.zeros((*nii_seg_sl.shape, 4)); rgba[...,0]=1; rgba[...,3]=nii_seg_sl*0.4
        axes[1, col].imshow(rgba, origin="lower")
    axes[1, col].set_title(f"NIfTI       z={z}", fontsize=8)
    axes[1, col].axis("off")

    # Row 2: NIfTI MR with raw DICOM SEG overlaid — checks if DICOM mask fits the NIfTI image
    axes[2, col].imshow(np.clip((nii_mr_sl-p2)/(p98-p2+1e-8),0,1).T, cmap="gray", origin="lower")
    if seg_sl.any():
        rgba = np.zeros((*seg_sl.shape, 4)); rgba[...,0]=0; rgba[...,1]=1; rgba[...,3]=seg_sl*0.4  # green
        axes[2, col].imshow(rgba, origin="lower")
    axes[2, col].set_title(f"NIfTI MR + DICOM SEG  z={z}", fontsize=8)
    axes[2, col].axis("off")

axes[0,0].set_ylabel("DICOM raw", fontsize=9)
axes[1,0].set_ylabel("NIfTI", fontsize=9)
axes[2,0].set_ylabel("NIfTI MR + DICOM SEG\n(green=DICOM mask)", fontsize=9)
plt.suptitle("Case: PCAMPMRI-00001_19940620\nRow 0: raw DICOM  |  Row 1: converted NIfTI  |  Row 2: cross-check", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Axis flip diagnosis: test flipped versions of the NIfTI SEG on the NIfTI MR ---
# If the converted seg has flipped axes, one of these should align perfectly.
nii_mr_sl  = img_nii_arr[:, :, 15]
nii_seg_sl = (seg_nii_arr[:, :, 15] > 0.5).astype(np.uint8)

candidates = {
    "original":      nii_seg_sl,
    "flip axis-0":   nii_seg_sl[::-1, :],
    "flip axis-1":   nii_seg_sl[:, ::-1],
    "flip both":     nii_seg_sl[::-1, ::-1],
    "transpose":     nii_seg_sl.T,
    "transpose+f0":  nii_seg_sl.T[::-1, :],
    "transpose+f1":  nii_seg_sl.T[:, ::-1],
}

p2, p98 = np.percentile(nii_mr_sl, [2, 98])
mr_disp = np.clip((nii_mr_sl - p2) / (p98 - p2 + 1e-8), 0, 1).T  # (W,H) for imshow

fig, axes = plt.subplots(1, len(candidates), figsize=(3*len(candidates), 4))
for ax, (label, seg) in zip(axes, candidates.items()):
    ax.imshow(mr_disp, cmap="gray", origin="lower")
    if seg.any():
        rgba = np.zeros((*seg.shape, 4)); rgba[...,0]=1; rgba[...,3]=seg*0.5
        ax.imshow(rgba, origin="lower")
    ax.set_title(label, fontsize=8)
    ax.axis("off")
plt.suptitle("NIfTI MR (slice 15) + flipped NIfTI SEG variants — find the one that aligns", fontsize=9)
plt.tight_layout()
plt.show()

In [35]:
# --- Inspect the affine math in _numpy_to_sitk more carefully ---
# The SEG is built as (D, H, W) numpy and passed to sitk.GetImageFromArray.
# SimpleITK interprets numpy (z, y, x) → sitk axes (x, y, z).
# The affine columns are [row_cos*col_sp, col_cos*row_sp, slice_cos*sl_sp].
# Column 0 = x direction = row_cos (direction of increasing column index = x in sitk).
# Column 1 = y direction = col_cos (direction of increasing row index = y in sitk).
# But in the numpy array, axis-1 (H) is rows, axis-2 (W) is cols.
# sitk maps numpy axis-2 → sitk x, numpy axis-1 → sitk y: this is correct.
# However: SetDirection uses flatten("F") = column-major, which gives
#   [col0[0..2], col1[0..2], col2[0..2]]
# SimpleITK direction is row-major: direction[i*3+j] = i-th basis vec component j.
# So SetDirection([x0,x1,x2, y0,y1,y2, z0,z1,z2]) maps axis-0 of sitk to x_dir, etc.
# flatten("F") on matrix whose columns are [x_dir, y_dir, z_dir] gives exactly this. ✓
#
# Potential bug: col_spacing and row_spacing are swapped in _seg_dicom_to_volumes:
#   col_spacing, row_spacing = pixel_spacing
# But DICOM PixelSpacing = [row_spacing, col_spacing] (spacing between rows = col direction spacing).
# For isotropic pixels this doesn't matter, but let's verify.

shared = seg_ds.SharedFunctionalGroupsSequence[0]
pixel_spacing = [float(v) for v in shared.PixelMeasuresSequence[0].PixelSpacing]
print(f"PixelSpacing from DICOM SEG SharedFunctionalGroupsSequence: {pixel_spacing}")
print(f"(DICOM standard: [row_spacing, col_spacing] = [between-rows, between-cols])")
print(f"In _seg_dicom_to_volumes: col_spacing={pixel_spacing[0]}, row_spacing={pixel_spacing[1]}")
print(f"Isotropic? {pixel_spacing[0] == pixel_spacing[1]}")

iop = [float(v) for v in shared.PlaneOrientationSequence[0].ImageOrientationPatient]
row_cos = np.array(iop[:3])
col_cos = np.array(iop[3:])
slice_cos = np.cross(row_cos, col_cos)
print(f"\nImageOrientationPatient row_cos: {np.round(row_cos,4)}")
print(f"ImageOrientationPatient col_cos: {np.round(col_cos,4)}")
print(f"slice_cos (cross product):       {np.round(slice_cos,4)}")

PixelSpacing from DICOM SEG SharedFunctionalGroupsSequence: [1.0156, 1.0156]
(DICOM standard: [row_spacing, col_spacing] = [between-rows, between-cols])
In _seg_dicom_to_volumes: col_spacing=1.0156, row_spacing=1.0156
Isotropic? True

ImageOrientationPatient row_cos: [ 0.9985  0.0543 -0.0046]
ImageOrientationPatient col_cos: [-0.054   0.9973  0.0498]
slice_cos (cross product):       [ 0.0073 -0.0495  0.9987]


In [36]:
# --- Reconstruct what _seg_dicom_to_volumes builds and compare to what SimpleITK MR reader gives ---
# SimpleITK MR (via GetGDCMSeriesFileNames) will reorder slices by ImagePositionPatient,
# which may differ from InstanceNumber order. Let's check.
mr_fnames = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(mr_series_dir))
mr_sitk = sitk.ReadImage(mr_fnames)
mr_sitk_arr = sitk.GetArrayFromImage(mr_sitk)  # (D, H, W) in z,y,x order
print(f"SimpleITK MR array shape: {mr_sitk_arr.shape}")
print(f"SimpleITK MR direction:\n{np.array(mr_sitk.GetDirection()).reshape(3,3)}")
print(f"SimpleITK MR spacing: {mr_sitk.GetSpacing()}")
print(f"SimpleITK MR origin:  {mr_sitk.GetOrigin()}")

SimpleITK MR array shape: (30, 512, 512)
SimpleITK MR direction:
[[ 0.9985 -0.054   0.0073]
 [ 0.0543  0.9973 -0.0495]
 [-0.0046  0.0498  0.9987]]
SimpleITK MR spacing: (0.3125, 0.3125, 2.999999501634917)
SimpleITK MR origin:  (-77.7007, -89.4624, -50.962)


In [37]:
# --- Compare SimpleITK MR slice order vs DICOM InstanceNumber order ---
# SimpleITK sorts by IPP projected onto slice normal (ascending).
# Our manual InstanceNumber sort may give different z ordering.
mr_sitk_zs = [mr_sitk.GetOrigin()[2] + i * mr_sitk.GetSpacing()[2] * mr_sitk.GetDirection()[8]
              for i in range(mr_sitk.GetDepth())]
print("SimpleITK MR z positions (first/last):", round(mr_sitk_zs[0],2), "→", round(mr_sitk_zs[-1],2))
print("Manual InstanceNumber z (first/last): ", round(mr_ipps[0][2],2), "→", round(mr_ipps[-1][2],2))
print("SEG z (first/last):                   ", round(wg_ipps[0][2],2), "→", round(wg_ipps[-1][2],2))

print(f"\nSimpleITK MR first-slice IPP from origin: {np.round(mr_sitk.GetOrigin(),2)}")
print(f"SEG first-slice IPP:                      {np.round(wg_ipps[0],2)}")

SimpleITK MR z positions (first/last): -50.96 → 35.93
Manual InstanceNumber z (first/last):  35.93 → -50.96
SEG z (first/last):                    -54.71 → 35.18

SimpleITK MR first-slice IPP from origin: [-77.7  -89.46 -50.96]
SEG first-slice IPP:                      [-124.6  -141.6   -54.71]


In [38]:
# --- Simulate _seg_dicom_to_volumes exactly and compare to the saved NIfTI ---
# Reproduce the code path from download_qin_prostate_repeatability.py to see
# what sitk image is produced, then check its origin vs the MR.
import sys
sys.path.insert(0, "../src")
from download_qin_prostate_repeatability import _seg_dicom_to_volumes, _numpy_to_sitk

seg_file_path = next(seg_series_dir.glob("*.dcm"))
volumes, affine_ras, shape = _seg_dicom_to_volumes(seg_file_path)
wg_seg_num = next(n for n, d in num_to_desc.items() if "Whole" in str(d) or "whole" in str(d).lower())
seg_sitk_rebuilt = _numpy_to_sitk(volumes[wg_seg_num], affine_ras)

print("Rebuilt SEG sitk:")
print(f"  origin:    {np.round(seg_sitk_rebuilt.GetOrigin(), 2)}")
print(f"  spacing:   {np.round(seg_sitk_rebuilt.GetSpacing(), 4)}")
print(f"  direction:\n{np.array(seg_sitk_rebuilt.GetDirection()).reshape(3,3).round(4)}")
print(f"  size:      {seg_sitk_rebuilt.GetSize()}")

print("\nReference MR sitk:")
print(f"  origin:    {np.round(mr_sitk.GetOrigin(), 2)}")
print(f"  spacing:   {np.round(mr_sitk.GetSpacing(), 4)}")
print(f"  direction:\n{np.array(mr_sitk.GetDirection()).reshape(3,3).round(4)}")
print(f"  size:      {mr_sitk.GetSize()}")

Rebuilt SEG sitk:
  origin:    [-124.6  -141.6   -54.71]
  spacing:   [1.0156 1.0156 5.9999]
  direction:
[[ 0.9985  0.0543 -0.0046]
 [-0.054   0.9973  0.0498]
 [ 0.0073 -0.0495  0.9987]]
  size:      (256, 256, 16)

Reference MR sitk:
  origin:    [-77.7  -89.46 -50.96]
  spacing:   [0.3125 0.3125 3.    ]
  direction:
[[ 0.9985 -0.054   0.0073]
 [ 0.0543  0.9973 -0.0495]
 [-0.0046  0.0498  0.9987]]
  size:      (512, 512, 30)


In [ ]:
# --- Resample SEG onto MR and visualize to see if it overlaps correctly ---
resampled_seg = sitk.Resample(
    seg_sitk_rebuilt, mr_sitk, sitk.Transform(),
    sitk.sitkNearestNeighbor, 0, seg_sitk_rebuilt.GetPixelID(),
)
resampled_arr = sitk.GetArrayFromImage(resampled_seg)  # (D, H, W)
print(f"Resampled SEG shape: {resampled_arr.shape}, nonzero voxels: {resampled_arr.sum()}")

# Show mid-slice of the resampled seg on the MR (both in sitk array = z,y,x order)
mid_z = resampled_arr.shape[0] // 2
# Find best z with most seg content
best_z = np.argmax(resampled_arr.sum(axis=(1,2)))
print(f"Best z slice (most seg content): {best_z}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, z in zip(axes, [best_z-3, best_z, best_z+3]):
    z = max(0, min(z, mr_sitk_arr.shape[0]-1))
    mr_sl  = mr_sitk_arr[z]
    seg_sl = resampled_arr[z]
    p2, p98 = np.percentile(mr_sl, [2, 98])
    ax.imshow(np.clip((mr_sl-p2)/(p98-p2+1e-8),0,1), cmap="gray", origin="upper")
    if seg_sl.any():
        rgba = np.zeros((*seg_sl.shape, 4)); rgba[...,0]=1; rgba[...,3]=seg_sl*0.5
        ax.imshow(rgba, origin="upper")
    ax.set_title(f"sitk arrays slice z={z}", fontsize=9)
    ax.axis("off")
plt.suptitle("Resampled SEG on MR — both from SimpleITK arrays (no nibabel)\nRed = wholegland mask", fontsize=9)
plt.tight_layout()
plt.show()

## 7. Root Cause: `flatten(order="F")` Transposes the Direction Matrix

`_numpy_to_sitk` builds the direction cosine matrix correctly (columns = axis directions in LPS), but calls `.flatten(order="F")` before passing to `SetDirection`. ITK expects **row-major** format, so `"F"` (column-major) effectively transposes the matrix, swapping rows↔columns. This makes the SEG appear to have a different physical orientation than the MR, and the resampler maps each SEG voxel to the wrong anatomical position.

In [40]:
import sys; sys.path.insert(0, "../src")
from download_qin_prostate_repeatability import _seg_dicom_to_volumes

seg_series_dir = next(
    d for d in IMAGES_DIR.iterdir()
    if any(True for f in d.glob("*.dcm")
           if str(pydicom.dcmread(str(f), stop_before_pixels=True).get("PatientID","")).strip() == "PCAMPMRI-00001"
           and str(pydicom.dcmread(str(f), stop_before_pixels=True).get("StudyDate","")).strip() == "19940620"
           and str(pydicom.dcmread(str(f), stop_before_pixels=True).get("Modality","")).strip() == "SEG")
)

seg_file_path = next(seg_series_dir.glob("*.dcm"))
volumes_raw, affine_ras_raw, _ = _seg_dicom_to_volumes(seg_file_path)

def make_seg_sitk(vol, affine_ras, flatten_order):
    ras_to_lps = np.diag([-1., -1., 1.])
    vv = affine_ras[:3, :3]
    sp = np.linalg.norm(vv, axis=0).tolist()
    d  = (ras_to_lps @ (vv / sp)).flatten(order=flatten_order).tolist()
    o  = (ras_to_lps @ affine_ras[:3, 3]).tolist()
    img = sitk.GetImageFromArray(vol)
    img.SetSpacing(sp); img.SetDirection(d); img.SetOrigin(o)
    return img

wg_vol = volumes_raw[wg_num]
seg_buggy = make_seg_sitk(wg_vol, affine_ras_raw, "F")
seg_fixed = make_seg_sitk(wg_vol, affine_ras_raw, "C")

d_mr    = np.array(mr_sitk.GetDirection()).reshape(3, 3)
d_buggy = np.array(seg_buggy.GetDirection()).reshape(3, 3)
d_fixed = np.array(seg_fixed.GetDirection()).reshape(3, 3)

print("MR direction (expected):")
print(d_mr.round(4))
print("\nSEG direction with flatten('F')  — BUG (transposed):")
print(d_buggy.round(4))
print("\nSEG direction with flatten('C')  — FIXED:")
print(d_fixed.round(4))
print(f"\nFixed matches MR? {np.allclose(d_fixed, d_mr, atol=1e-3)}")

MR direction (expected):
[[ 0.9985 -0.054   0.0073]
 [ 0.0543  0.9973 -0.0495]
 [-0.0046  0.0498  0.9987]]

SEG direction with flatten('F')  — BUG (transposed):
[[ 0.9985  0.0543 -0.0046]
 [-0.054   0.9973  0.0498]
 [ 0.0073 -0.0495  0.9987]]

SEG direction with flatten('C')  — FIXED:
[[ 0.9985 -0.054   0.0073]
 [ 0.0543  0.9973 -0.0495]
 [-0.0046  0.0498  0.9987]]

Fixed matches MR? True


In [ ]:
# --- Visualize: buggy vs fixed resample on MR (sitk array space, no nibabel) ---
res_buggy = sitk.Resample(seg_buggy, mr_sitk, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
res_fixed = sitk.Resample(seg_fixed, mr_sitk, sitk.Transform(), sitk.sitkNearestNeighbor, 0)

arr_buggy = sitk.GetArrayFromImage(res_buggy)   # (D,H,W)
arr_fixed  = sitk.GetArrayFromImage(res_fixed)
mr_arr    = sitk.GetArrayFromImage(mr_sitk)

# Pick slices with most fixed-mask content
best_zs = np.argsort(arr_fixed.sum(axis=(1,2)))[-4:][::-1]

fig, axes = plt.subplots(3, len(best_zs), figsize=(3*len(best_zs), 9))
for col, z in enumerate(sorted(best_zs)):
    mr_sl = mr_arr[z]
    p2, p98 = np.percentile(mr_sl, [2, 98])
    mr_norm = np.clip((mr_sl - p2)/(p98 - p2 + 1e-8), 0, 1)

    for row, (arr, color_ch, title) in enumerate([
        (None,      None, f"MR only  z={z}"),
        (arr_buggy, 0,    f"+ buggy SEG  z={z}"),
        (arr_fixed, 1,    f"+ fixed SEG  z={z}"),
    ]):
        ax = axes[row, col]
        ax.imshow(mr_norm, cmap="gray", origin="upper")
        if arr is not None and arr[z].any():
            rgba = np.zeros((*arr[z].shape, 4))
            rgba[..., color_ch] = 1.0    # 0=red (buggy), 1=green (fixed)
            rgba[..., 3] = arr[z] * 0.45
            ax.imshow(rgba, origin="upper")
        ax.set_title(title, fontsize=8)
        ax.axis("off")

axes[0, 0].set_ylabel("MR only", fontsize=9)
axes[1, 0].set_ylabel("Buggy SEG (red)\nflatten='F'", fontsize=9)
axes[2, 0].set_ylabel("Fixed SEG (green)\nflatten='C'", fontsize=9)
plt.suptitle("Case PCAMPMRI-00001_19940620 — Bug: flatten('F') transposes direction matrix\nFix: flatten('C') = correct ITK row-major format", fontsize=9)
plt.tight_layout()
plt.show()